# 01 — Exploratory Data Analysis
## Telco Customer Churn Dataset

**Objective:** Understand the dataset structure, distributions, and initial churn patterns before building models.

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

import sys
sys.path.append('..')
from src.preprocessing import load_data, create_demographic_features, create_service_features, create_contract_features, create_payment_features, create_spend_features
from src.preprocessing import DATA_PATH

In [ ]:
df = load_data()
print(f"Shape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

## 2. Data Overview

In [ ]:
df.head(3)

In [ ]:
df.info()

In [ ]:
df.describe()

### Missing Values

In [ ]:
missing = df.isnull().sum()
display(missing[missing > 0] if missing.sum() > 0 else "No missing values found after cleaning.")

## 3. Target Variable — Churn

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100
churn_summary = pd.DataFrame({'Count': churn_counts, 'Percentage': churn_pct.round(2)})
churn_summary

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
churn_counts.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Churn Count')
axes[0].set_ylabel('Customers')
axes[0].tick_params(axis='x', rotation=0)

axes[1].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Churn Proportion')
plt.tight_layout()
plt.savefig('../reports/churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Demographic Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
cat_demo = ['gender', 'SeniorCitizen', 'Partner', 'Dependents']
for i, col in enumerate(cat_demo):
    ax = axes[i // 3, i % 3]
    ctab = pd.crosstab(df[col], df['Churn'], normalize='index') * 100
    ctab.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'], rot=0)
    ax.set_title(f'Churn Rate by {col}')
    ax.set_ylabel('Percentage')
    ax.legend(title='Churn')

axes[1, 2].axis('off')
plt.tight_layout()
plt.savefig('../reports/demographic_churn.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Tenure Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist([df[df['Churn']=='No']['tenure'],
              df[df['Churn']=='Yes']['tenure']],
             bins=30, label=['Stayed', 'Churned'],
             color=['#2ecc71', '#e74c3c'], alpha=0.7)
axes[0].set_xlabel('Tenure (months)')
axes[0].set_ylabel('Customers')
axes[0].set_title('Tenure Distribution by Churn')
axes[0].legend()

df['TenureBin'] = pd.cut(df['tenure'], bins=[0, 6, 12, 24, 48, 72],
                          labels=['0-6mo', '6-12mo', '1-2yr', '2-4yr', '4-6yr'])
churn_by_tenure = df.groupby('TenureBin', observed=True)['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100)
churn_by_tenure.plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_title('Churn Rate by Tenure Group')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('../reports/tenure_churn.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Contract & Payment Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
contract_churn = df.groupby('Contract')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100)
contract_churn.plot(kind='bar', ax=axes[0], color=['#e74c3c', '#f39c12', '#2ecc71'])
axes[0].set_title('Churn Rate by Contract Type')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].tick_params(axis='x', rotation=0)

payment_churn = df.groupby('PaymentMethod')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100)
payment_churn.sort_values().plot(kind='barh', ax=axes[1], color='teal')
axes[1].set_title('Churn Rate by Payment Method')
axes[1].set_xlabel('Churn Rate (%)')
plt.tight_layout()
plt.savefig('../reports/contract_payment_churn.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Service Analysis

In [ ]:
service_cols = ['PhoneService', 'MultipleLines', 'InternetService',
                    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                    'TechSupport', 'StreamingTV', 'StreamingMovies']
service_churn = []
for col in service_cols:
    rate = df.groupby(col, observed=True)['Churn'].apply(
        lambda x: (x == 'Yes').mean() * 100)
    for cat, val in rate.items():
        service_churn.append({'Service': col, 'Category': cat, 'ChurnRate': val})
service_df = pd.DataFrame(service_churn)

plt.figure(figsize=(14, 6))
sns.barplot(data=service_df, x='Service', y='ChurnRate', hue='Category')
plt.xticks(rotation=45)
plt.title('Churn Rate by Service Type')
plt.ylabel('Churn Rate (%)')
plt.tight_layout()
plt.savefig('../reports/service_churn.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Monthly Charges Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist([df[df['Churn']=='No']['MonthlyCharges'],
              df[df['Churn']=='Yes']['MonthlyCharges']],
             bins=30, label=['Stayed', 'Churned'],
             color=['#2ecc71', '#e74c3c'], alpha=0.7)
axes[0].set_xlabel('Monthly Charges ($)')
axes[0].set_ylabel('Customers')
axes[0].set_title('Monthly Charges Distribution by Churn')
axes[0].legend()

df['ChargeBin'] = pd.qcut(df['MonthlyCharges'], q=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
charge_churn = df.groupby('ChargeBin', observed=True)['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100)
charge_churn.plot(kind='bar', ax=axes[1], color='purple')
axes[1].set_title('Churn Rate by Monthly Charges Quintile')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('../reports/charges_churn.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Correlation Analysis

In [ ]:
df_encoded = pd.get_dummies(df.select_dtypes(include=['object']), drop_first=True)
df_numeric = pd.concat([df.select_dtypes(include=[np.number]), df_encoded], axis=1)
corr = df_numeric.corr()
churn_corr = corr['Churn_Yes'].sort_values(key=abs, ascending=False)
churn_corr_top = churn_corr.head(15)

plt.figure(figsize=(10, 8))
sns.heatmap(corr[churn_corr_top.index].loc[churn_corr_top.index],
            annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Top 15 Churn Correlations')
plt.tight_layout()
plt.savefig('../reports/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

churn_corr_top

## 10. Key Findings

| Finding | Insight |
|---------|---------|
| **Overall churn rate** | ~26.5% of customers churned |
| **Contract type** | Month-to-month contracts have ~3x higher churn than one-year, ~5x higher than two-year |
| **Tenure** | Churn rate drops sharply after 12 months; highest in first 6 months |
| **Payment method** | Electronic check users churn at ~2x the rate of automatic payment users |
| **Services** | Lack of online security and tech support strongly correlates with churn |
| **Internet service** | Fiber optic customers churn more than DSL (likely due to higher competition) |
| **Demographics** | Seniors and customers without partners/dependents churn more

---
*End of 01 — Exploratory Data Analysis*